In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_1.py ──
"""
Shared infrastructure for MLFP02 Exercise 1 — Probability and Bayesian
Fundamentals.

Contains: HDB data loading (4-ROOM 2020+ slice), prior/posterior math for
the Normal-Normal and Beta-Binomial conjugate families, bootstrap helpers,
and output directory wiring.

Technique-specific narrative (MLE derivation, Bayes scenarios, credible vs
confidence simulation, bootstrap comparison) does NOT belong here — it
lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for plots (HTML) — every technique writes here.
OUTPUT_DIR = Path("outputs") / "mlfp02_ex1_bayes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical prior used across all four technique files. A single source
# of truth means Task 5's prior sweep and Task 4's posterior use the same
# starting point.
PRIOR_MU_0: float = 500_000.0  # SGD — Singapore 4-room HDB market anchor
PRIOR_SIGMA_0: float = 100_000.0  # SGD — moderate uncertainty

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (Singapore, data.gov.sg)
# ════════════════════════════════════════════════════════════════════════

_loader = MLFPDataLoader()


def load_hdb_all() -> pl.DataFrame:
    """Full HDB resale dataset filtered to 2020+ transactions.

    Returns a polars DataFrame with columns: month, town, flat_type,
    resale_price, plus any others present in the source parquet. Used
    by Task 1 (truth tables) and Task 8 (expected value by flat type).
    """
    hdb = _loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def load_hdb_4room() -> pl.DataFrame:
    """4-ROOM slice of HDB resale (2020+) — the primary estimation target."""
    return load_hdb_all().filter(pl.col("flat_type") == "4 ROOM")


def load_hdb_prices_4room() -> np.ndarray:
    """Return the 4-room resale_price column as a float64 numpy array.

    This is the primary observation vector for MLE, Normal-Normal, and
    bootstrap tasks.
    """
    return load_hdb_4room()["resale_price"].to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# NORMAL-NORMAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalPosterior:
    """Posterior for μ under a Normal-Normal conjugate with known σ."""

    mean: float
    std: float
    prior_mean: float
    prior_std: float
    n: int
    sigma_known: float

    @property
    def precision_prior(self) -> float:
        return 1.0 / self.prior_std**2

    @property
    def precision_data(self) -> float:
        return self.n / self.sigma_known**2

    @property
    def prior_weight(self) -> float:
        """Fraction of posterior precision contributed by the prior (0..1)."""
        return self.precision_prior / (self.precision_prior + self.precision_data)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        z = stats.norm.ppf(0.5 + level / 2)
        return (self.mean - z * self.std, self.mean + z * self.std)


def normal_normal_posterior(
    data: np.ndarray,
    prior_mean: float,
    prior_std: float,
    sigma_known: float,
) -> NormalPosterior:
    """Closed-form posterior for μ under N(μ₀, σ₀²) prior and known σ.

    Posterior precision = prior precision + n / σ². Posterior mean is the
    precision-weighted average of the prior mean and the sample mean.
    """
    n = len(data)
    xbar = float(np.mean(data))
    prec_prior = 1.0 / prior_std**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    sigma_post_sq = 1.0 / prec_post
    mu_post = sigma_post_sq * (prior_mean * prec_prior + n * xbar / sigma_known**2)
    return NormalPosterior(
        mean=mu_post,
        std=float(np.sqrt(sigma_post_sq)),
        prior_mean=prior_mean,
        prior_std=prior_std,
        n=n,
        sigma_known=sigma_known,
    )


# ════════════════════════════════════════════════════════════════════════
# BETA-BINOMIAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class BetaPosterior:
    """Posterior for p under a Beta-Binomial conjugate."""

    alpha: float
    beta: float
    prior_alpha: float
    prior_beta: float
    k: int
    n: int

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def prior_mean(self) -> float:
        return self.prior_alpha / (self.prior_alpha + self.prior_beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        lo = (1 - level) / 2
        hi = 1 - lo
        return tuple(stats.beta.ppf([lo, hi], self.alpha, self.beta).tolist())


def beta_binomial_posterior(
    k: int, n: int, prior_alpha: float, prior_beta: float
) -> BetaPosterior:
    """Closed-form posterior for p under Beta(α, β) prior and k/n Binomial."""
    return BetaPosterior(
        alpha=prior_alpha + k,
        beta=prior_beta + (n - k),
        prior_alpha=prior_alpha,
        prior_beta=prior_beta,
        k=k,
        n=n,
    )


# ════════════════════════════════════════════════════════════════════════
# MLE + CRAMER-RAO
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalMLE:
    mean: float
    mle_std: float  # ddof=0, biased
    unbiased_std: float  # ddof=1, Bessel's correction
    n: int

    @property
    def fisher_information(self) -> float:
        return self.n / self.mle_std**2

    @property
    def cramer_rao_bound(self) -> float:
        return 1.0 / self.fisher_information

    @property
    def standard_error(self) -> float:
        return float(np.sqrt(self.cramer_rao_bound))


def normal_mle(data: np.ndarray) -> NormalMLE:
    """MLE for Normal (μ, σ²) with Cramér-Rao bound bookkeeping."""
    arr = np.asarray(data, dtype=np.float64)
    return NormalMLE(
        mean=float(arr.mean()),
        mle_std=float(arr.std(ddof=0)),
        unbiased_std=float(arr.std(ddof=1)),
        n=len(arr),
    )


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP INTERVALS
# ════════════════════════════════════════════════════════════════════════


def bootstrap_mean_distribution(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42
) -> np.ndarray:
    """Non-parametric bootstrap of the sample mean."""
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array(
        [rng.choice(data, size=n, replace=True).mean() for _ in range(n_bootstrap)]
    )


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    lo = (1 - level) / 2 * 100
    hi = (1 + level) / 2 * 100
    return float(np.percentile(draws, lo)), float(np.percentile(draws, hi))


def bca_ci(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42, level: float = 0.95
) -> tuple[float, float]:
    """Bias-corrected accelerated bootstrap CI for the mean (via scipy)."""
    result = stats.bootstrap(
        (np.asarray(data, dtype=np.float64),),
        statistic=np.mean,
        n_resamples=n_bootstrap,
        confidence_level=level,
        method="BCa",
        random_state=seed,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


# ════════════════════════════════════════════════════════════════════════
# FORMATTING
# ════════════════════════════════════════════════════════════════════════


def fmt_money(x: float) -> str:
    return f"${x:,.0f}"


def print_interval(label: str, lo: float, hi: float) -> None:
    print(
        f"  {label:<28} [{fmt_money(lo)}, {fmt_money(hi)}]  width={fmt_money(hi - lo)}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 1.2: Bayes' Theorem — From Medical Tests to
#                         Property Valuation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Apply Bayes' theorem P(A|B) = P(B|A)P(A)/P(B) to real scenarios
#   - Understand the base rate fallacy — why a 99.5% specificity test
#     still produces many false positives at low prevalence
#   - Sweep prevalence to see how posterior probability changes
#   - Apply Bayesian reasoning to Singapore HDB property valuation
#   - Quantify dollar impact of ignoring base rates in decision-making
#
# PREREQUISITES: Complete 01_probability_mle.py (joint/conditional probs)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why Bayes' theorem matters for real-world decisions
#   2. Build — COVID ART test: sensitivity, specificity, prevalence
#   3. Train — prevalence sweep shows base rate fallacy in action
#   4. Visualise — posterior probability vs prevalence curve
#   5. Apply — HDB Bishan valuation: is a $650K listing overpriced?
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots



## THEORY — Why Bayes' Theorem Matters for Real-World Decisions


In [ ]:
# Bayes' theorem:
#   P(A|B) = P(B|A) × P(A) / P(B)
#
# In plain language: to find the probability of A given that B happened,
# you need THREE things:
#   1. P(B|A) — the likelihood (how likely is B if A is true?)
#   2. P(A) — the prior (how common is A before any observation?)
#   3. P(B) — the marginal (how common is B overall?)
#
# The base rate fallacy: people overweight the test result and underweight
# the base rate P(A). A COVID test with 99.5% specificity sounds nearly
# perfect, but when prevalence is only 2%, most positive results are false
# positives. The cure: always ask "how common is the condition?"



## TASK 2 — BUILD: COVID ART Test — Sensitivity, Specificity, Prevalence


In [ ]:
print("\n" + "=" * 70)
print("  MLFP02 Exercise 1.2: Bayes' Theorem Applications")
print("=" * 70)

# COVID ART test parameters (Singapore community testing context)
sensitivity = 0.85  # P(positive test | infected)
specificity = 0.995  # P(negative test | not infected)
prevalence = 0.02  # P(infected) — base rate in Singapore community

# TODO: Compute P(positive test) using the law of total probability
# P(+) = P(+|infected)×P(infected) + P(+|not infected)×P(not infected)
# Hint: sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
p_positive = ____

# TODO: Compute P(infected | positive test) using Bayes' theorem
# Hint: (sensitivity * prevalence) / p_positive
p_infected_given_positive = ____

# P(not infected | positive test) — false positive rate among positives
p_false_positive = 1 - p_infected_given_positive

print(f"\n=== Bayes' Theorem: COVID ART Test ===")
print(f"Sensitivity: {sensitivity:.1%} — P(+test | infected)")
print(f"Specificity: {specificity:.1%} — P(-test | not infected)")
print(f"Prevalence:  {prevalence:.1%} — P(infected)")
print(f"")
print(f"P(positive test)           = {p_positive:.4f} ({p_positive:.2%})")
print(
    f"P(infected | positive test) = {p_infected_given_positive:.4f} "
    f"({p_infected_given_positive:.1%})"
)
print(f"P(false positive)           = {p_false_positive:.4f} ({p_false_positive:.1%})")
# INTERPRETATION: Even with a 99.5% specificity test, when prevalence is
# only 2%, a positive test means you're truly infected only ~77% of the
# time. This is the base rate fallacy.



### Checkpoint 1


In [ ]:
assert 0 < p_infected_given_positive < 1, "Posterior probability must be valid"
assert (
    p_infected_given_positive > prevalence
), "Positive test must increase probability of infection above base rate"
print("\n✓ Checkpoint 1 passed — Bayes' theorem computed\n")



## TASK 3 — TRAIN: Prevalence Sweep — Base Rate Fallacy in Action


In [ ]:
print("--- Effect of Prevalence on P(infected | +test) ---")
prevalence_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50]
posterior_values = []

for prev in prevalence_values:
    # TODO: Compute P(positive) and P(infected | positive) for each prevalence
    # Hint: same Bayes formula as above, but with prev instead of prevalence
    p_pos = ____
    p_inf = ____
    posterior_values.append(p_inf)
    print(f"  Prevalence {prev:>5.1%} → P(infected | +) = {p_inf:.1%}")

# INTERPRETATION: At 0.1% prevalence (non-epidemic), a positive test means
# you're only ~15% likely to be infected. At 50% (outbreak peak), ~99.4%.



### Checkpoint 2


In [ ]:
assert len(posterior_values) == len(prevalence_values), "One posterior per prevalence"
assert (
    posterior_values[0] < posterior_values[-1]
), "Higher prevalence → higher posterior"
assert all(0 < p < 1 for p in posterior_values), "All posteriors must be valid"
print("\n✓ Checkpoint 2 passed — prevalence sweep complete\n")



## TASK 4 — VISUALISE: Posterior Probability vs Prevalence Curve


In [ ]:
# Fine-grained sweep for smooth curve
prev_fine = np.linspace(0.001, 0.5, 200)
# TODO: Compute posterior for each prevalence in prev_fine
# Hint: list comprehension with the Bayes formula
post_fine = [____ for p in prev_fine]

fig1 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "P(infected | positive test) vs Prevalence",
        "False Positive Rate vs Prevalence",
    ],
)

# Left: posterior probability
fig1.add_trace(
    go.Scatter(
        x=prev_fine * 100,
        y=np.array(post_fine) * 100,
        mode="lines",
        name="P(infected | +test)",
        line={"color": "red", "width": 2},
    ),
    row=1,
    col=1,
)
fig1.add_trace(
    go.Scatter(
        x=[p * 100 for p in prevalence_values],
        y=[p * 100 for p in posterior_values],
        mode="markers",
        name="Computed points",
        marker={"color": "red", "size": 8},
    ),
    row=1,
    col=1,
)
fig1.add_hline(
    y=50,
    line_dash="dash",
    line_color="gray",
    row=1,
    col=1,
    annotation_text="50% — coin flip",
)

# Right: false positive rate
fp_fine = [1 - p for p in post_fine]
fig1.add_trace(
    go.Scatter(
        x=prev_fine * 100,
        y=np.array(fp_fine) * 100,
        mode="lines",
        name="False positive rate",
        line={"color": "blue", "width": 2},
    ),
    row=1,
    col=2,
)

fig1.update_xaxes(title_text="Prevalence (%)", row=1, col=1)
fig1.update_xaxes(title_text="Prevalence (%)", row=1, col=2)
fig1.update_yaxes(title_text="Posterior (%)", row=1, col=1)
fig1.update_yaxes(title_text="False Positive Rate (%)", row=1, col=2)
fig1.update_layout(title="Base Rate Fallacy: Why Test Results Mislead", height=450)
fig1.write_html(str(OUTPUT_DIR / "bayes_prevalence_sweep.html"))
print("Saved: bayes_prevalence_sweep.html")



### Checkpoint 3


In [ ]:
assert len(post_fine) == 200, "Fine sweep should have 200 points"
print("\n✓ Checkpoint 3 passed — visualisations saved\n")



## TASK 5 — APPLY: HDB Bishan Valuation — Is a $650K Listing Overpriced?


In [ ]:
# Scenario: a 4-room flat in Bishan is listed at $650K. A buyer asks:
# "What fraction of similar flats actually sell above $600K?"

print("=== APPLICATION: HDB Bishan Valuation ===")

hdb_4room = load_hdb_4room()
# TODO: Filter for Bishan flats
# Hint: hdb_4room.filter(pl.col("town") == "BISHAN")
bishan_flats = ____

if bishan_flats.height > 0:
    # TODO: Compute fraction of Bishan flats with price > $600K
    # Hint: filter for resale_price > 600_000, divide height by total
    p_above_600k_bishan = ____
    mean_bishan = bishan_flats["resale_price"].mean()
    print(f"Bishan 4-room data: {bishan_flats.height} transactions")
    print(f"Mean price: {fmt_money(mean_bishan)}")
    print(f"P(price > $600K | Bishan 4-room) = {p_above_600k_bishan:.2%}")
    print(f"This empirical probability is the 'data-driven prior' for Bishan.")

    # Dollar impact analysis
    listing_price = 650_000
    if p_above_600k_bishan > 0.5:
        print(f"\n→ {p_above_600k_bishan:.0%} of Bishan 4-room flats sell above $600K.")
        print(f"  A $650K listing is within the normal range for this location.")
        overpay_risk = listing_price - mean_bishan
        if overpay_risk > 0:
            print(
                f"  But at {fmt_money(abs(overpay_risk))} above the mean, negotiate down."
            )
        else:
            print(f"  At {fmt_money(abs(overpay_risk))} below the mean — good deal.")
    else:
        print(f"\n→ Only {p_above_600k_bishan:.0%} sell above $600K.")
        print(f"  $650K is above the market norm — negotiate or walk away.")
else:
    print("No Bishan 4-room data found — using uninformative estimate")

# Compare Bishan vs market-wide
prices_array = load_hdb_prices_4room()
market_p_above_600k = float((prices_array > 600_000).mean())
print(f"\nMarket-wide P(4-room > $600K) = {market_p_above_600k:.2%}")
if bishan_flats.height > 0:
    lift = p_above_600k_bishan / market_p_above_600k if market_p_above_600k > 0 else 0
    print(f"Bishan premium: {lift:.2f}x the market-wide rate")



### Checkpoint 4


In [ ]:
assert market_p_above_600k >= 0, "Market probability must be non-negative"
print("\n✓ Checkpoint 4 passed — Bishan valuation complete\n")



## REFLECTION


✓ Bayes' theorem: P(A|B) = P(B|A)P(A)/P(B) — the engine behind
    every Bayesian update
  ✓ Base rate fallacy: a 99.5% specificity test is unreliable when
    prevalence is low — most positives are false positives
  ✓ Prevalence sweep: posterior probability is a smooth function of
    the base rate — visualised for stakeholder communication
  ✓ Real-world application: Bishan property valuation using
    conditional probabilities from actual transaction data
  ✓ Dollar framing: translate statistical results into actionable
    pricing decisions with explicit overpay risk

  NEXT: In 03_conjugate_priors.py, you'll implement the Normal-Normal
  and Beta-Binomial conjugate families — the mathematical machinery
  that makes Bayesian updating computationally tractable.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED (1.2 — Bayes' Theorem)")
print("═" * 70)
print(
)

print("\n✓ Exercise 1.2 complete — Bayes' Theorem")

